# DAR cold-start SFT on Kaggle (clean run)

Notebook này được sắp xếp để chạy ổn định bằng **Save & Run All**: cài package trước mọi import, không dùng DeepSpeed, kiểm tra đủ video trước khi train, và mặc định chỉ chạy smoke test 2 steps.

Các input cần attach trước khi chạy:

- `vucongaaa/dar-r1`: repo DAR và offline wheelhouse.
- `vucongaaa/dar-annotation`: `train.jsonl`.
- `vucongaaa/vce-original-videos`: video VCE.
- Kaggle Model `qwen-lm/qwen2.5-vl/transformers/3b-instruct/2`.

Không thêm import `numpy`, `pandas`, `torch`, `transformers`, `swift` hoặc `kagglehub` trước cell cài wheelhouse.

In [ ]:
%%bash
set -euo pipefail

WHEELHOUSE=/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128
REPO_SOURCE=/kaggle/input/datasets/vucongaaa/dar-r1/DAR
ANNOTATION=/kaggle/input/datasets/vucongaaa/dar-annotation/train.jsonl
VIDEO_DATASET=/kaggle/input/datasets/vucongaaa/vce-original-videos
MODEL_SOURCE=/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2

test -d "$WHEELHOUSE"
test -f "$WHEELHOUSE/ms_swift-3.12.5-py3-none-any.whl"
test -d "$REPO_SOURCE"
test -f "$ANNOTATION"
test -d "$VIDEO_DATASET"
test -f "$MODEL_SOURCE/config.json"
test -f "$MODEL_SOURCE/preprocessor_config.json"

echo "All required Kaggle inputs are attached."
python --version
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
set -euo pipefail

WHEELHOUSE=/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128

RUNTIME=/kaggle/working/dar_python
mkdir -p "$RUNTIME"

# Không ghi đè /usr/local: Kaggle có thể đã nạp NumPy trong kernel.
mapfile -d '' WHEELS < <(find "$WHEELHOUSE" -maxdepth 1 -type f \
  -name '*.whl' \
  ! -name '*cp313*' \
  ! -name 'packaging-26.3*' \
  -print0 | sort -z)

test "${#WHEELS[@]}" -gt 0
python -m pip install \
  --no-index --no-deps --upgrade \
  --target "$RUNTIME" \
  "${WHEELS[@]}"

test -x "$RUNTIME/bin/swift"
echo "Isolated offline runtime ready: $RUNTIME"

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/kaggle/working/dar_python

# Kiểm tra trong process mới, không import package vào kernel notebook.
python - <<'PY'
import sys
import torch
import transformers
import swift
import trl
import peft
import datasets
import pyarrow
import modelscope
import json_repair
import qwen_vl_utils
import decord
import numpy
import scipy

print("Python:", sys.version.split()[0])
print("numpy:", numpy.__version__, numpy.__file__)
print("scipy:", scipy.__version__)
print("torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("transformers:", transformers.__version__)
print("swift:", swift.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("pyarrow:", pyarrow.__version__)
print("modelscope:", modelscope.__version__)
assert transformers.__version__ == "4.57.1"
assert swift.__version__ == "3.12.5"
assert torch.cuda.is_available(), "Kaggle GPU is not enabled"
PY

In [ ]:
%%bash
set -euo pipefail

SOURCE=/kaggle/input/datasets/vucongaaa/dar-r1/DAR
TARGET=/kaggle/working/DAR

# Idempotent: chạy lại cell không tạo /DAR/DAR.
mkdir -p "$TARGET"
cp -a "$SOURCE"/. "$TARGET"/

LAUNCHER=$TARGET/ms-swift/examples/train/sft/dar/train_qwen2.5vl_sft_dar.sh
PREPARE=$TARGET/ms-swift/examples/train/sft/dar/prepare_dar_sft_data.py

# Repair older dar-r1 inputs whose helper was indented by four spaces.
if head -n 1 "$PREPARE" | grep -q '^    #!/usr/bin/env python3'; then
  sed -i 's/^    //' "$PREPARE"
fi
python -m py_compile "$PREPARE"

# RTX 6000 96 GB chạy single-GPU không cần DeepSpeed.
sed -i '/^[[:space:]]*--deepspeed "${DEEPSPEED}" \\/d' "$LAUNCHER"

if grep -q -- '--deepspeed' "$LAUNCHER"; then
  echo "Failed to remove --deepspeed from launcher" >&2
  exit 1
fi

echo "Writable DAR repo ready: $TARGET"
echo "DeepSpeed disabled in launcher."

In [ ]:
from pathlib import Path
import json
import os

source = Path("/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2")
target = Path("/kaggle/working/qwen2_5_vl_3b_fixed")
target.mkdir(parents=True, exist_ok=True)

# Symlink weights; chỉ tạo bản writable cho hai JSON metadata nhỏ.
for src in source.iterdir():
    if src.name in {"config.json", "preprocessor_config.json"}:
        continue
    dst = target / src.name
    if not os.path.lexists(dst):
        dst.symlink_to(src, target_is_directory=src.is_dir())

with (source / "config.json").open(encoding="utf-8") as f:
    config = json.load(f)
config["model_type"] = "qwen2_5_vl"
config.setdefault("architectures", ["Qwen2_5_VLForConditionalGeneration"])
with (target / "config.json").open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

with (source / "preprocessor_config.json").open(encoding="utf-8") as f:
    processor_config = json.load(f)
processor_config["image_processor_type"] = "Qwen2VLImageProcessor"
processor_config["processor_class"] = "Qwen2_5_VLProcessor"
with (target / "preprocessor_config.json").open("w", encoding="utf-8") as f:
    json.dump(processor_config, f, ensure_ascii=False, indent=2)

print("Model overlay ready:", target)

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/kaggle/working/dar_python

python - <<'PY'
from transformers import AutoProcessor

model_path = "/kaggle/working/qwen2_5_vl_3b_fixed"
processor = AutoProcessor.from_pretrained(
    model_path,
    trust_remote_code=True,
    local_files_only=True,
)
print("Processor:", type(processor))
PY

In [ ]:
%%bash
set -euo pipefail

VIDEO_DATASET=/kaggle/input/datasets/vucongaaa/vce-original-videos
ANNOTATION=/kaggle/input/datasets/vucongaaa/dar-annotation/train.jsonl
OUTPUT=/kaggle/working/dar_sft/train_qwen25vl_ms_sft.jsonl
PREPARE=/kaggle/working/DAR/ms-swift/examples/train/sft/dar/prepare_dar_sft_data.py

SAMPLE_VIDEO=$(find "$VIDEO_DATASET" -type f -name '06720.mp4' -print -quit)
test -n "$SAMPLE_VIDEO" || {
  echo "Cannot find 06720.mp4 under $VIDEO_DATASET" >&2
  exit 1
}

VIDEO_ROOT=$(dirname "$SAMPLE_VIDEO")
mkdir -p /kaggle/working/dar_sft
echo "VIDEO_ROOT=$VIDEO_ROOT"

# Backward-compatible repair for the older dar-r1 helper.
if head -n 1 "$PREPARE" | grep -q '^    #!/usr/bin/env python3'; then
  sed -i 's/^    //' "$PREPARE"
fi
python -m py_compile "$PREPARE"

python "$PREPARE" \
  --input "$ANNOTATION" \
  --output "$OUTPUT" \
  --video-root "$VIDEO_ROOT" \
  --check-videos \
  --strict

test "$(wc -l < "$OUTPUT")" -eq 13646
echo "Validated all 13,646 SFT records and video paths."

## Train

Giữ `MODE=smoke` khi kiểm tra hoặc dùng Save & Run All lần đầu. Sau khi smoke test hoàn tất 2 steps, đổi thành `MODE=full` để chạy 0.5 epoch. Full mode chỉ lưu model weights và giữ một checkpoint để output không vượt giới hạn lưu trữ của Kaggle.

In [ ]:
%%bash
set -euo pipefail

MODE=smoke  # Đổi thành full sau khi smoke test thành công.

cd /kaggle/working/DAR/ms-swift/examples/train/sft/dar

export PYTHONPATH=/kaggle/working/dar_python
export PATH=/kaggle/working/dar_python/bin:$PATH
export CUDA_VISIBLE_DEVICES=0
export NPROC_PER_NODE=1
export SWIFT_BIN=/kaggle/working/dar_python/bin/swift
export GRADIENT_ACCUMULATION_STEPS=32
export MODEL_NAME=/kaggle/working/qwen2_5_vl_3b_fixed
export DATA_JSONL=/kaggle/working/dar_sft/train_qwen25vl_ms_sft.jsonl
export PREPARE_DATA=0

EXTRA_ARGS=(--model_type qwen2_5_vl)

case "$MODE" in
  smoke)
    export OUTPUT_DIR=/kaggle/working/dar_sft/smoke
    EXTRA_ARGS+=(--max_steps 2 --save_strategy no)
    ;;
  full)
    export OUTPUT_DIR=/kaggle/working/dar_sft/checkpoints
    EXTRA_ARGS+=(--save_only_model true --save_total_limit 1)
    ;;
  *)
    echo "MODE must be smoke or full" >&2
    exit 1
    ;;
esac

echo "Training mode: $MODE"
echo "Output: $OUTPUT_DIR"
bash train_qwen2.5vl_sft_dar.sh "${EXTRA_ARGS[@]}"

In [ ]:
%%bash
set -euo pipefail

echo "Artifacts under /kaggle/working/dar_sft:"
du -h -d 2 /kaggle/working/dar_sft | sort -h
find /kaggle/working/dar_sft -maxdepth 3 -type f \
  \( -name 'trainer_state.json' -o -name 'config.json' -o -name '*.safetensors' \) \
  -print